# Practice 4-7. LLM 기반 리랭킹 (Reranking) — 로컬 Qwen3-Reranker-0.6B

책은 FAISS 임베딩 검색(`k=4`) 결과를 `gpt-4o` 기반 LLM 리랭커로 상위 `top_n=2`개로 압축합니다.

`04-postgresql-bm25-sparse-retrieval.ipynb`(BM25)·`05-pgvector-hnsw-dense-retrieval.ipynb`(pgvector 밀집 검색)·`06-bm25-pgvector-ensemble-retrieval.ipynb`(앙상블)에서 이미 문서 522건에 대한 색인을
pgvector 서버에 갖춰 두었으므로, 이 노트북은 임베딩 인프라를 새로 만들지 않고 **그대로 재사용**합니다
(다른 노트북들과 동일하게 이 노트북 안에서 처음부터 다시 구축하므로, 앞선 노트북을 먼저 실행해 두지 않아도 됩니다).

`06-bm25-pgvector-ensemble-retrieval.ipynb`의 마지막 관찰에서 지적했듯, `EnsembleRetriever`는 두 리트리버 각각의 상위 k건을 순위 기반으로 합칠 뿐
재순위화하지는 않습니다 — "실무에서는 각 리트리버의 후보 수를 최종 반환 개수보다 넉넉히 크게 잡아 합친 뒤 재순위화(rerank)하는
방식으로 이 한계를 완화합니다"라고 예고했던 부분이 이 노트북의 주제입니다. 각 리트리버의 후보 폭을 `k=4`로 넓혀 합친 뒤,
**[Qwen3-Reranker-0.6B](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B)** 로컬 모델로 최종 `top_n=2`를 정밀하게 골라냅니다.

## 0. 환경 설정

In [ ]:
import os
import re
import torch
import psycopg
from urllib.parse import quote_plus

# Hugging Face 인증이 필요하면 HF_TOKEN 환경변수를 사용한다.
# --- LM Studio (1차 후보 검색용 임베딩 + 최종 답변 생성 LLM) ---
LMSTUDIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://...:.../v1")
LMSTUDIO_API_KEY = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

EMBED_MODEL = "embedding-8b:sl"
LLM_MODEL   = "qwen3.6-35b:mm"

# --- 로컬 Qwen3 리랭커 (2차 재순위화만 담당, 임베딩은 위 LM Studio 모델을 그대로 사용) ---
RERANKER_MODEL_NAME = "Qwen/Qwen3-Reranker-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- PostgreSQL (pgvector 이미지, mecab-ko/textsearch_ko 확장 포함, Docker 컨테이너) ---
PG_HOST = os.getenv("PG_HOST", "pgvector")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB = os.getenv("PG_DB", "admin")
PG_USER = os.getenv("PG_USER", "admin")
PG_PASSWORD = os.getenv("PG_PASSWORD", "...")
PG_DSN = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
# langchain_postgres 는 SQLAlchemy 비동기 엔진(asyncpg)을 쓰므로 URL 형식으로도 준비해 둔다.
PG_ASYNC_URL = f"postgresql+asyncpg://{PG_USER}:{quote_plus(PG_PASSWORD)}@{PG_HOST}:{PG_PORT}/{PG_DB}"

# --- 입력 파일 (노트북 기준 상대경로) ---
DATA_PATH = "practice4/투자설명서.pdf"


def strip_think(text: str) -> str:
    """qwen3 등 추론 모델의 <think>...</think> 블록 제거."""
    if text is None:
        return ""
    return re.sub(r"(^|<think>).*?</think>", "", text, flags=re.DOTALL).strip()


print("device:", DEVICE)

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    temperature=0,
    # qwen3 계열 추론 모델은 답변 전에 <think> 추론 토큰을 많이 소모하므로 넉넉히 잡는다
    max_tokens=2048,
)

# check_embedding_ctx_length=False : tiktoken 대신 원문 문자열을 전송 → LM Studio 호환
# chunk_size=8 : 로컬 LM Studio 서버가 한 번에 받는 텍스트 수가 많으면 연결이 끊기므로 작게 배치
base_embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    check_embedding_ctx_length=False,
    chunk_size=8,
)

with psycopg.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        print("PostgreSQL:", cur.fetchone()[0].split(",")[0])

print("LLM  :", strip_think(llm.invoke("연결 테스트. 'ok' 라고만 답하세요.").content)[:50])

## 1. 문서 로드 · 분할

앞선 검색 노트북들과 동일하게 `투자설명서.pdf`를 `chunk_size=2000, chunk_overlap=200`으로 분할합니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(DATA_PATH)
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

docs = loader.load_and_split(doc_splitter)
print("분할된 문서 수:", len(docs))

QUESTION = "이 회사가 발행한 주식의 총 발행량이 어느 정도야?"

## 2. 검색 인프라 재구축 — 희소 검색(BM25)

`04-postgresql-bm25-sparse-retrieval.ipynb`/`06-bm25-pgvector-ensemble-retrieval.ipynb`과 동일한 스키마·트리거·BM25 함수입니다(자세한 설명은 `04-postgresql-bm25-sparse-retrieval.ipynb` 참고). `DELETE` 후
재적재하므로 재실행해도 안전합니다.

In [ ]:
DDL_BM25_ALL = """
CREATE EXTENSION IF NOT EXISTS textsearch_ko;

CREATE TABLE IF NOT EXISTS practice4_bm25_docs (
    id       SERIAL PRIMARY KEY,
    content  TEXT NOT NULL,
    page     INTEGER,
    tsv      TSVECTOR,
    doc_len  INTEGER
);
CREATE INDEX IF NOT EXISTS idx_practice4_bm25_docs_tsv
    ON practice4_bm25_docs USING GIN (tsv);

CREATE TABLE IF NOT EXISTS practice4_bm25_term_freq (
    doc_id  INTEGER NOT NULL REFERENCES practice4_bm25_docs(id) ON DELETE CASCADE,
    token   TEXT NOT NULL,
    tf      INTEGER NOT NULL,
    PRIMARY KEY (doc_id, token)
);
CREATE INDEX IF NOT EXISTS idx_practice4_bm25_term_freq_token
    ON practice4_bm25_term_freq (token);

CREATE TABLE IF NOT EXISTS practice4_bm25_term_df (
    token  TEXT PRIMARY KEY,
    df     INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS practice4_bm25_stats (
    id         SMALLINT PRIMARY KEY DEFAULT 1,
    doc_count  INTEGER NOT NULL DEFAULT 0,
    total_len  BIGINT  NOT NULL DEFAULT 0,
    CHECK (id = 1)
);
INSERT INTO practice4_bm25_stats (id, doc_count, total_len)
VALUES (1, 0, 0)
ON CONFLICT (id) DO NOTHING;

CREATE OR REPLACE FUNCTION practice4_bm25_docs_before() RETURNS TRIGGER AS $$
BEGIN
    NEW.tsv := to_tsvector('korean', NEW.content);
    SELECT COALESCE(SUM(cardinality(positions)), 0)
      INTO NEW.doc_len
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights);
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_before
    BEFORE INSERT OR UPDATE ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_before();

CREATE OR REPLACE FUNCTION practice4_bm25_docs_after_insert() RETURNS TRIGGER AS $$
BEGIN
    INSERT INTO practice4_bm25_term_freq (doc_id, token, tf)
    SELECT NEW.id, lexeme, cardinality(positions)
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (doc_id, token) DO UPDATE SET tf = EXCLUDED.tf;

    INSERT INTO practice4_bm25_term_df (token, df)
    SELECT lexeme, 1
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (token) DO UPDATE SET df = practice4_bm25_term_df.df + 1;

    UPDATE practice4_bm25_stats
       SET doc_count = doc_count + 1,
           total_len = total_len + NEW.doc_len
     WHERE id = 1;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_after_insert
    AFTER INSERT ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_after_insert();

CREATE OR REPLACE FUNCTION practice4_bm25_docs_after_delete() RETURNS TRIGGER AS $$
BEGIN
    DELETE FROM practice4_bm25_term_freq WHERE doc_id = OLD.id;

    UPDATE practice4_bm25_term_df
       SET df = df - 1
     WHERE token IN (SELECT lexeme FROM unnest(OLD.tsv) AS u(lexeme, positions, weights));

    DELETE FROM practice4_bm25_term_df WHERE df <= 0;

    UPDATE practice4_bm25_stats
       SET doc_count = doc_count - 1,
           total_len = total_len - OLD.doc_len
     WHERE id = 1;
    RETURN OLD;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_after_delete
    AFTER DELETE ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_after_delete();

CREATE OR REPLACE FUNCTION practice4_bm25_docs_after_update() RETURNS TRIGGER AS $$
BEGIN
    DELETE FROM practice4_bm25_term_freq WHERE doc_id = OLD.id;

    UPDATE practice4_bm25_term_df
       SET df = df - 1
     WHERE token IN (SELECT lexeme FROM unnest(OLD.tsv) AS u(lexeme, positions, weights));

    DELETE FROM practice4_bm25_term_df WHERE df <= 0;

    UPDATE practice4_bm25_stats SET doc_count = doc_count - 1, total_len = total_len - OLD.doc_len WHERE id = 1;

    INSERT INTO practice4_bm25_term_freq (doc_id, token, tf)
    SELECT NEW.id, lexeme, cardinality(positions)
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (doc_id, token) DO UPDATE SET tf = EXCLUDED.tf;

    INSERT INTO practice4_bm25_term_df (token, df)
    SELECT lexeme, 1
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (token) DO UPDATE SET df = practice4_bm25_term_df.df + 1;

    UPDATE practice4_bm25_stats SET doc_count = doc_count + 1, total_len = total_len + NEW.doc_len WHERE id = 1;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_after_update
    AFTER UPDATE ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_after_update();

CREATE OR REPLACE FUNCTION practice4_bm25_search(
    q       text,
    k       integer DEFAULT 5,
    bm25_k1 float8  DEFAULT 1.2,
    bm25_b  float8  DEFAULT 0.75
) RETURNS TABLE(doc_id integer, content text, page integer, score float8) AS $$
    WITH stats AS (
        SELECT doc_count::float8 AS n,
               CASE WHEN doc_count > 0 THEN total_len::float8 / doc_count ELSE 0 END AS avgdl
          FROM practice4_bm25_stats WHERE id = 1
    ),
    qtokens AS (
        SELECT DISTINCT lexeme AS token
          FROM unnest(to_tsvector('korean', q)) AS u(lexeme, positions, weights)
    ),
    qidf AS (
        SELECT qt.token,
               ln( (s.n - COALESCE(d.df, 0) + 0.5) / (COALESCE(d.df, 0) + 0.5) + 1 ) AS idf
          FROM qtokens qt
          CROSS JOIN stats s
          LEFT JOIN practice4_bm25_term_df d ON d.token = qt.token
    ),
    scored AS (
        SELECT tf.doc_id,
               SUM( qidf.idf * (tf.tf * (bm25_k1 + 1))
                    / (tf.tf + bm25_k1 * (1 - bm25_b + bm25_b * doc.doc_len / NULLIF(s.avgdl, 0)))
                  ) AS score
          FROM practice4_bm25_term_freq tf
          JOIN qidf ON qidf.token = tf.token
          JOIN practice4_bm25_docs doc ON doc.id = tf.doc_id
          CROSS JOIN stats s
         GROUP BY tf.doc_id
    )
    SELECT scored.doc_id, doc.content, doc.page, scored.score
      FROM scored
      JOIN practice4_bm25_docs doc ON doc.id = scored.doc_id
     ORDER BY scored.score DESC
     LIMIT k;
$$ LANGUAGE sql STABLE;
"""

with psycopg.connect(PG_DSN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_BM25_ALL)
print("BM25 스키마 준비 완료")

In [ ]:
def reload_bm25_docs():
    """재실행 안전: DELETE → 트리거가 파생 테이블(순색인/역색인/통계)을 원복 → 재적재."""
    with psycopg.connect(PG_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute("DELETE FROM practice4_bm25_docs;")
            cur.execute("ALTER SEQUENCE practice4_bm25_docs_id_seq RESTART WITH 1;")
            for d in docs:
                cur.execute(
                    "INSERT INTO practice4_bm25_docs (content, page) VALUES (%s, %s)",
                    (d.page_content, d.metadata.get("page")),
                )
        conn.commit()

    with psycopg.connect(PG_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT doc_count, total_len FROM practice4_bm25_stats;")
            return cur.fetchone()


bm25_doc_count, bm25_total_len = reload_bm25_docs()
assert bm25_doc_count == len(docs), "BM25 적재 건수가 분할 문서 수와 다릅니다"
print("BM25 적재 완료:", bm25_doc_count, "건")

## 3. 검색 인프라 재구축 — 밀집 검색(pgvector HNSW)

`05-pgvector-hnsw-dense-retrieval.ipynb`/`06-bm25-pgvector-ensemble-retrieval.ipynb`과 동일하게 `embedding-8b:sl`(4096차원, MRL 학습)을 앞쪽 1024차원만 잘라 pgvector HNSW
인덱스 상한(2000차원) 안에 맞춥니다. `overwrite_existing=True`로 테이블을 매번 새로 만들기 때문에 재실행해도 안전합니다.

In [ ]:
from langchain_core.embeddings import Embeddings

PROJ_DIM = 1024  # pgvector HNSW(vector) 인덱스 상한(2000차원)보다 충분히 작게


class TruncatedEmbeddings(Embeddings):
    """LM Studio 임베딩(embedding-8b:sl, MRL로 학습된 Qwen3 계열)의 앞쪽 out_dim개 차원만 사용해
    pgvector HNSW 인덱스 상한 안으로 맞추는 래퍼 (자세한 설명은 05-pgvector-hnsw-dense-retrieval.ipynb 참고)."""

    def __init__(self, base: Embeddings, out_dim: int):
        self.base = base
        self.out_dim = out_dim

    def embed_documents(self, texts: list) -> list:
        return [v[: self.out_dim] for v in self.base.embed_documents(texts)]

    def embed_query(self, text: str) -> list:
        return self.base.embed_query(text)[: self.out_dim]


embeddings = TruncatedEmbeddings(base_embeddings, out_dim=PROJ_DIM)

from langchain_postgres import PGEngine, PGVectorStore
from langchain_postgres.v2.indexes import HNSWIndex, DistanceStrategy

DENSE_TABLE = "practice4_dense_docs"

engine = PGEngine.from_connection_string(url=PG_ASYNC_URL)


def rebuild_dense_table():
    """재실행 안전: DROP TABLE IF EXISTS 후 재생성 → 재적재 → HNSW 재인덱싱."""
    engine.init_vectorstore_table(
        table_name=DENSE_TABLE,
        vector_size=PROJ_DIM,
        overwrite_existing=True,
    )
    vs = PGVectorStore.create_sync(
        engine=engine,
        embedding_service=embeddings,
        table_name=DENSE_TABLE,
        distance_strategy=DistanceStrategy.COSINE_DISTANCE,
        k=4,
    )
    vs.add_documents(docs)
    vs.apply_vector_index(
        HNSWIndex(name="practice4_dense_docs_hnsw", distance_strategy=DistanceStrategy.COSINE_DISTANCE, m=16, ef_construction=64)
    )
    with psycopg.connect(PG_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT count(*) FROM {DENSE_TABLE};")
            row_count = cur.fetchone()[0]
    return vs, row_count


vectorstore, dense_doc_count = rebuild_dense_table()
assert dense_doc_count == len(docs), "밀집 검색 적재 건수가 분할 문서 수와 다릅니다"
print("밀집 검색 인덱스 준비 완료:", dense_doc_count, "건, HNSW 유효성:", vectorstore.is_valid_index("practice4_dense_docs_hnsw"))

### 3.1 재실행 안전성 직접 증명 — 두 인프라를 한 번씩 더 초기화·재구축

`06-bm25-pgvector-ensemble-retrieval.ipynb`과 동일하게, 말로만 안전하다고 하지 않고 두 인프라를 한 번씩 더 재구축해서 값이 그대로인지 확인합니다.

In [ ]:
bm25_doc_count_2nd, bm25_total_len_2nd = reload_bm25_docs()
vectorstore, dense_doc_count_2nd = rebuild_dense_table()

print("BM25      : 1차", (bm25_doc_count, bm25_total_len), " / 2차", (bm25_doc_count_2nd, bm25_total_len_2nd))
print("밀집 검색 : 1차", dense_doc_count, " / 2차", dense_doc_count_2nd)

assert (bm25_doc_count, bm25_total_len) == (bm25_doc_count_2nd, bm25_total_len_2nd)
assert dense_doc_count == dense_doc_count_2nd
print("\n두 인프라 모두 재구축 결과가 이전과 정확히 일치합니다 — 이 노트북은 몇 번을 다시 실행해도 안전합니다.")

## 4. 앙상블 1차 후보 검색기 (후보 폭 k=4)

`06-bm25-pgvector-ensemble-retrieval.ipynb`은 `bm25_retriever`·`dense_retriever` 각각 `k=2`로 바로 최종 결과를 만들었습니다. 이 노트북은 최종 반환
개수(`top_n=2`)보다 넉넉한 `k=4`로 각 리트리버의 후보를 넓게 뽑은 뒤, 다음 절의 리랭커로 정밀하게 좁힙니다.

In [ ]:
from typing import List
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.callbacks import CallbackManagerForRetrieverRun


class PgSqlFunctionRetriever(BaseRetriever):
    """PostgreSQL의 practice4_bm25_search() 함수를 호출해 검색하는 리트리버 (04-postgresql-bm25-sparse-retrieval.ipynb 참고)."""

    dsn: str
    sql_func: str
    k: int = 4

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        with psycopg.connect(self.dsn) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    f"SELECT doc_id, content, page, score FROM {self.sql_func}(%s, %s)",
                    (query, self.k),
                )
                rows = cur.fetchall()
        return [
            Document(
                page_content=content,
                metadata={"source": DATA_PATH, "doc_id": doc_id, "page": page, "score": float(score)},
            )
            for doc_id, content, page, score in rows
        ]


bm25_retriever = PgSqlFunctionRetriever(dsn=PG_DSN, sql_func="practice4_bm25_search", k=4)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

try:
    from langchain_classic.retrievers import EnsembleRetriever
except ImportError:
    from langchain.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
)

candidates = ensemble_retriever.invoke(QUESTION)

# 두 리트리버가 같은 청크를 함께 뽑을 수 있으므로 리랭커에 넘기기 전 중복 제거
seen, unique_candidates = set(), []
for d in candidates:
    if d.page_content not in seen:
        seen.add(d.page_content)
        unique_candidates.append(d)

print("앙상블 후보 수:", len(candidates), "/ 중복 제거 후:", len(unique_candidates))
for d in unique_candidates:
    print(" -", d.metadata.get("page"), "|", d.page_content[:50].replace("\n", " "))

## 5. Qwen3-Reranker-0.6B 로컬 리랭커

[모델 카드](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B)의 `AutoModelForCausalLM` 예제를 따릅니다.
질의-문서 쌍을 `<Instruct>/<Query>/<Document>` 프롬프트로 구성한 뒤, 모델이 다음 토큰으로 `yes`/`no`를
생성할 로그확률을 관련성 점수로 사용합니다.

> 참고: `sentence-transformers`의 `CrossEncoder("Qwen/Qwen3-Reranker-0.6B")` 로딩 방식도 모델 카드에 있지만,
> 실제로는 분류 헤드(`score.weight`)가 체크포인트에 없어 무작위 초기화되어 점수가 무의미합니다.
> 따라서 모델 카드의 원조 방식인 `AutoModelForCausalLM` 기반 yes/no 스코어링을 사용합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


class Qwen3Reranker:
    """Qwen3-Reranker-0.6B: 질의-문서 쌍에 대해 yes/no 다음 토큰 확률을 관련성 점수로 사용."""

    def __init__(self, model_name: str = RERANKER_MODEL_NAME, device: str = DEVICE):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to(device).eval()

        self.token_false_id = self.tokenizer.convert_tokens_to_ids("no")
        self.token_true_id = self.tokenizer.convert_tokens_to_ids("yes")
        self.max_length = 8192

        self.prefix = (
            "<|im_start|>system\nJudge whether the Document meets the requirements based on "
            "the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\"."
            "<|im_end|>\n<|im_start|>user\n"
        )
        self.suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
        self.prefix_tokens = self.tokenizer.encode(self.prefix, add_special_tokens=False)
        self.suffix_tokens = self.tokenizer.encode(self.suffix, add_special_tokens=False)
        self.task = "주어진 검색 질의에 대해 답변이 될 수 있는 관련 문서를 찾아라"

    def _format(self, query: str, doc: str) -> str:
        return f"<Instruct>: {self.task}\n<Query>: {query}\n<Document>: {doc}"

    def _process(self, pairs: List[str]):
        inputs = self.tokenizer(
            pairs, padding=False, truncation="longest_first", return_attention_mask=False,
            max_length=self.max_length - len(self.prefix_tokens) - len(self.suffix_tokens),
        )
        for i, ele in enumerate(inputs["input_ids"]):
            inputs["input_ids"][i] = self.prefix_tokens + ele + self.suffix_tokens
        inputs = self.tokenizer.pad(inputs, padding=True, return_tensors="pt", max_length=self.max_length)
        return {k: v.to(self.model.device) for k, v in inputs.items()}

    @torch.no_grad()
    def score(self, query: str, doc_texts: List[str]) -> List[float]:
        pairs = [self._format(query, d) for d in doc_texts]
        inputs = self._process(pairs)
        batch_scores = self.model(**inputs).logits[:, -1, :]
        true_vector = batch_scores[:, self.token_true_id]
        false_vector = batch_scores[:, self.token_false_id]
        stacked = torch.stack([false_vector, true_vector], dim=1)
        probs = torch.nn.functional.log_softmax(stacked, dim=1)
        return probs[:, 1].exp().tolist()


reranker = Qwen3Reranker()


def reranking_documents(query: str, docs_: List[Document], top_n: int = 2) -> List[Document]:
    scores = reranker.score(query, [d.page_content for d in docs_])
    scored_docs = sorted(zip(docs_, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in scored_docs[:top_n]]

## 6. 검색 확인 — 리랭킹 전/후 비교

In [ ]:
reranked_docs = reranking_documents(QUESTION, unique_candidates, top_n=2)

print(f"Query: {QUESTION}\n")
print(f"앙상블 후보 {len(unique_candidates)}건 → 리랭킹 후 상위 {len(reranked_docs)}건\n")
for i, doc in enumerate(reranked_docs):
    print(f"Document {i + 1} (page={doc.metadata.get('page')}):")
    print(doc.page_content[:200])
    print()

## 7. RerankingRetriever로 래핑

앙상블 검색기를 1차 후보 검색기로 감싸고, 그 결과를 리랭커로 좁히는 `BaseRetriever`를 정의합니다.

In [ ]:
from typing import Any
from pydantic import ConfigDict, Field


class RerankingRetriever(BaseRetriever):
    """1차 후보 검색기(base_retriever)의 결과를 Qwen3 리랭커로 재순위화해 top_n건만 반환."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    base_retriever: Any = Field(description="1차 후보 검색기 (앙상블)")
    reranker_top_n: int = 2

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        candidates_ = self.base_retriever.invoke(query)
        seen_, unique_ = set(), []
        for d in candidates_:
            if d.page_content not in seen_:
                seen_.add(d.page_content)
                unique_.append(d)
        return reranking_documents(query, unique_, top_n=self.reranker_top_n)


reranking_retriever = RerankingRetriever(base_retriever=ensemble_retriever, reranker_top_n=2)
print("retriever 준비 완료:", type(reranking_retriever).__name__)

## 8. 답변 생성 (RetrievalQA)

In [ ]:
try:
    from langchain_classic.chains import RetrievalQA
except ImportError:
    from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=reranking_retriever,
    return_source_documents=True,
)

result = qa_chain.invoke({"query": QUESTION})

print("질문:", QUESTION)
print("답변:", strip_think(result["result"]))
print("\n사용된 문서:")
for d in result["source_documents"]:
    print(" -", d.metadata)

## 9. 관찰 — 리랭킹은 앙상블의 "합치기만 하는" 한계를 보완한다

`06-bm25-pgvector-ensemble-retrieval.ipynb`의 `EnsembleRetriever`(각 리트리버 `k=2`)는 BM25와 밀집 검색의 상위 2건씩을 순위 기반(RRF)으로 그대로
합쳤을 뿐, 합쳐진 결과 자체를 다시 평가하지는 않았습니다. 이 노트북은 각 리트리버의 후보 폭을 `k=4`로 넓혀 더 많은 후보를
모은 뒤, Qwen3-Reranker-0.6B가 질의와 각 후보 문서를 실제로 다시 비교해 `top_n=2`를 정밀하게 골라냅니다.

그 결과 최종 답변은 `05-pgvector-hnsw-dense-retrieval.ipynb`/`06-bm25-pgvector-ensemble-retrieval.ipynb`과 동일하게 **13,602,977주**를 정확히 찾아냈습니다 — 다만 이번에는
"각 리트리버의 상위 k건이 우연히 정답을 포함했는가"에 의존하지 않고, 더 넓은 후보 폭(k=4×2=최대 8건) 중에서 리랭커가
실제 관련성을 기준으로 골라냈다는 점이 다릅니다. 이는 두 리트리버의 상위 k건 안에 정답이 없을 가능성이 있는 실무
상황에서 특히 유효한 전략입니다.